# PDF Parsing and Ingestion

In [ ]:
from langchain_community.document_loaders import (
    PyPDFLoader,
    PyMuPDFLoader    
)

### 1. PyPdfLoader

In [ ]:
try:
    pypdf_loader = PyPDFLoader("data/pdf_files/text-only.pdf")
    pypdfDocument = pypdf_loader.load();
    # print(pypdfDocument)

    print(f"✅ Successfully loaded file. -{pypdfDocument[0].metadata.get("source")}")
    for i, doc in enumerate(pypdfDocument, start=1):
        page_num = doc.metadata.get("page", "Unknown page")
        content_preview = doc.page_content[:200]  # first 200 chars

        print(f"Page {i}: Page Length: {len(doc.page_content)}")    
except Exception as e:
    print(f"Error: {e}")

### 2. PyMuPdfLoader

In [ ]:
try:
    pypdf_mu_loader = PyMuPDFLoader("data/pdf_files/text-only.pdf")
    pypdfDocument = pypdf_mu_loader.load();
    # print(pypdfDocument)

    print(f"✅ Successfully loaded file. -{pypdfDocument[0].metadata.get("source")}")
    for i, doc in enumerate(pypdfDocument, start=1):
        page_num = doc.metadata.get("page", "Unknown page")
        content_preview = doc.page_content[:200]  # first 200 chars

        print(f"Page {i}: Page Length: {len(doc.page_content)}")    
except Exception as e:
    print(f"Error: {e}")

### 📊 PyPdfLoader Vs PyMuPdfLoader

| Feature                   | PyPDFLoader                  | PyMuPDFLoader              |
|---------------------------|------------------------------|----------------------------|
| ✅ Reliability            | Simple and reliable          | Fast processing            |
| ✅ Suitability            | Good for most PDFs           | Good text extraction       |
| ✅ Page Handling          | Preserves page numbers       | Image extraction support   |
| ❌ Limitation             | Basic text extraction        | —                          |
| **Best Use Case**         | Standard text PDFs            | When speed is important    |

### Handling PDF Challenges

🎯 Purpose of This Section
PDFs are notoriously difficult to parse because they:

- Store text in complex ways - Table, Images, Special Chars
- Can have formatting issues
- May contain scanned images (requiring OCR)
- Often have extraction artifacts

In [ ]:
## Raw Text
raw_pdf_text = """Company Financial Report


    The ﬁnancial performance for ﬁscal year 2024
    shows signiﬁcant growth in proﬁtability.
    
    
    
    Revenue increased by 25%.
    
The company's efﬁciency improved due to workﬂow
optimization.


Page 1 of 10
"""


## clean text function
def cleanText(textToClean):
    # remove extra spaces
    cleanedText = " ".join(textToClean.split())
    # Fix ligatures
    cleanedText = cleanedText.replace("ﬁ", "fi")
    cleanedText = cleanedText.replace("ﬂ", "fl")
    return cleanedText;


cleanedText = cleanText(raw_pdf_text);
print("BEFORE:")
print(repr(raw_pdf_text))
print("\nAFTER:")
print(repr(cleanedText))

### PdfProcessor

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from typing import List

c:\zorkzpace\ai-datascience\RAGUdemy\RAGUdemyPoc\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class SmartPDFProcessor:
    """Advanced Pdf Processor"""
    #Constructor
    def __init__(self,chunk_size=1000,chunk_overlap=100):
        self.text_splitter=RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=[" "],
        )

    # Core function to process the pdf
    def process_pdf(self,pdf_path:str)->List[Document]:
        """Process PDF with smart chunking and metadata enhancement"""
        
        # 1. Laod PDF
        #######################################################################
        pdfLoader = PyPDFLoader(pdf_path);

        # 2. Extract the Pages and Process each Page
        #######################################################################
        pages = pdfLoader.load();

        #3. Process each Page
        #######################################################################
        processedChunks = []
        for page_num,page in enumerate(pages):
            ## clean text
            cleaned_text=self._clean_text(page.page_content)
            # Skip nearly empty pages
            if len(cleaned_text.strip()) < 50:
                continue
            
            # We use create_documents to split text into chunks while preserving metadata in Document objects for downstream processing.
            chunks = self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(cleaned_text)
                }]
            )
            processedChunks.extend(chunks)            
        return processedChunks;

    
    
    def _clean_text(self, text: str) -> str:
        """Clean extracted text"""
        # Remove excessive whitespace
        text = " ".join(text.split())
        
        # Fix common PDF extraction issues
        text = text.replace("ﬁ", "fi")
        text = text.replace("ﬂ", "fl")
        
        return text        


In [6]:
# Stub to test above code
pdfProcessorObj = SmartPDFProcessor();
print(type(pdfProcessorObj))

try:
   filePath = "data/pdf_files/transformer.pdf";
   smartChunks = pdfProcessorObj.process_pdf(filePath)
   print(f"Processed into {len(smartChunks)} smart chunks")
   # Show enhanced metadata
   if smartChunks:
      print("\nSample chunk metadata:")
      for key, value in smartChunks[0].metadata.items():
         print(f"  {key}: {value}")
except Exception as e:
    print(f"Processing error: {e}")

<class '__main__.SmartPDFProcessor'>
Processed into 67 smart chunks

Sample chunk metadata:
  producer: pdfTeX-1.40.21
  creator: LaTeX with hyperref
  creationdate: 2023-03-12T19:58:14+00:00
  author: Benyamin Ghojogh, Ali Ghodsi
  keywords: Tutorial
  moddate: 2023-03-12T19:58:14+00:00
  ptex.fullbanner: This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2
  subject: Proceedings of the International Conference on Machine Learning 2016
  title: Attention Mechanism, Transformers, BERT, and GPT: Tutorial and Survey
  trapped: /False
  source: data/pdf_files/transformer.pdf
  total_pages: 14
  page: 1
  page_label: 1
  chunk_method: smart_pdf_processor
  char_count: 3312
